In [1]:
import sys, os
sys.path.insert(0, '/home/okonias/projects/td-mpc_o2')
sys.path.insert(0, '/home/okonias/projects/td-mpc_o2/tdmpc/src')
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['MUJOCO_GL'] = 'egl'

In [2]:
import torch
import numpy as np
import time
from pathlib import Path
from omegaconf import OmegaConf
from cfg import parse_cfg
from env import make_env
from algorithm.helper import Episode, ReplayBuffer, linear_schedule
from o2.tdmpc_o2 import TDMPC_O2
from o2.training_utils import set_seed, update_tdmpc, update_decoder
from o2.eval_utils import evaluate_agent

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [4]:
O2_DEFAULTS = {
    'latent_action_dim': 128, 'decoder_init': False, 'use_latent_state': True,
    'dcem_batch_size': 64, 'decoder_updates': 100, 'told_updates': 500,
    'decoder_start_steps': 5000, 'exp_name': 'o2_ddpg',
    'latent_num_samples': 32, 'latent_num_elites': 8, 'dcem_sampling_n': None,
}

CFG_PATH = Path('/home/okonias/projects/td-mpc_o2/tdmpc/cfgs')
sys.argv = ['train', 'task=cheetah-run', 'seed=20']
cfg = parse_cfg(CFG_PATH)
cfg = OmegaConf.merge(OmegaConf.create(O2_DEFAULTS), cfg)

ar = cfg.action_repeat  # already set from task yaml
cfg = OmegaConf.merge(cfg, OmegaConf.create({
    'iterations' : 5,
    'told_updates' : 500,
    'decoder_updates': 80,

    # evaluation
    'eval_episodes': 0,
    'video_mode':    'none',

    # training — divided by action_repeat so wall-clock steps are consistent
    'train_steps':  40000 // ar,
    'seed_steps':   4000 // ar,   #CHANGE THIS
    'episode_length':   1000 // ar,

    # schedules — end step also divided by action_repeat
    'horizon_schedule':  f'linear(1, ${{horizon}}, {25000 // ar})',     #AND THIs
    'std_schedule':      f'linear(0.5, ${{min_std}}, {25000 // ar})',

    'decoder_start_steps':  0 // ar,
    'dcem_batch_size':   128,         #CHANGE THIS

    'use_latent_state': False,
    'decoder_init':   None,

     #'load_model': "models/cheetah-run20k/model.pt",
     #'load_buffer':  "models/cheetah-run20k/replay_buffer.pth",

    'momentum'          : 0.1,
    'use_is_weights'    : True,
    'lml_temperature'   : 1,
    'dec_grad_clip_norm': 1,

    'latent_action_dim': 128,
    'lambda_gae':        0.5,
     #'diversity_coeff':   0.01,
    'log_det_target' :   -8,
    'use_raw_obs': True,
     #'diversity_coeff_schedule': f'linear(0.1, 0.01, {10000 // ar})',

}))
# Override anything here:
# cfg.lr = 3e-4
# cfg.decoder_start_steps = 2000

print(OmegaConf.to_yaml(cfg))

latent_action_dim: 128
decoder_init: null
use_latent_state: false
dcem_batch_size: 128
decoder_updates: 80
told_updates: 500
decoder_start_steps: 0
exp_name: default
latent_num_samples: 32
latent_num_elites: 8
dcem_sampling_n: null
task: cheetah-run
modality: state
action_repeat: 4
discount: 0.99
episode_length: 250
train_steps: 10000
iterations: 5
num_samples: 512
num_elites: 64
mixture_coef: 0.05
min_std: 0.05
temperature: 0.5
momentum: 0.1
batch_size: 512
max_buffer_size: 1000000
horizon: 5
reward_coef: 0.5
value_coef: 0.1
consistency_coef: 2
rho: 0.5
kappa: 0.1
lr: 0.001
std_schedule: linear(0.5, ${min_std}, 6250)
horizon_schedule: linear(1, ${horizon}, 6250)
per_alpha: 0.6
per_beta: 0.4
grad_clip_norm: 10
seed_steps: 1000
update_freq: 2
tau: 0.01
enc_dim: 256
mlp_dim: 512
latent_dim: 50
use_wandb: false
wandb_project: none
wandb_entity: none
seed: 20
eval_freq: 20000
eval_episodes: 0
save_video: false
save_model: false
task_title: Cheetah Run
device: cuda
video_mode: none
use_is_w

In [7]:
set_seed(cfg.seed)
env    = make_env(cfg)
agent  = TDMPC_O2(cfg)
buffer = ReplayBuffer(cfg)
step, episode_idx = 0, 0
start_time = time.time()
print('Ready')

Ready


In [39]:
if cfg.get('load_model', None):
    d = torch.load(cfg.load_model)
    state_dict = d['model'] if 'model' in d else d
    for k in [k for k in state_dict if k.startswith('_action_decoder') or k.startswith('_V')]:
        del state_dict[k]
    agent.model.load_state_dict(state_dict, strict=False)
    if 'model_target' in d:
        target_dict = d['model_target']
        for k in [k for k in target_dict if k.startswith('_action_decoder') or k.startswith('_V')]:
            del target_dict[k]
        agent.model_target.load_state_dict(target_dict, strict=False)
        print("Target loaded")
    else:
        agent.model_target.load_state_dict(agent.model.state_dict(), strict=False)
        print("Target copied")
    print(f'Loaded model from {cfg.load_model}')
if cfg.get('load_buffer', None):
    buffer.__dict__.update(torch.load(cfg.load_buffer, weights_only=False))
    print(f'Loaded buffer from {cfg.load_buffer}')


In [17]:

# --- Decoder pretraining ---
agent.cfg.decoder_updates = 300
dec_metrics = update_decoder(agent, buffer, cfg, step=0)
agent.cfg.decoder_updates =  100

for k, v in dec_metrics.items():
    if k != 'grad_tracker':
        print(f'  {k:<20}: {v:>8.4f}')

  decoder_loss        : -25.2813
  decoder_grad_norm   :   1.0592
  value_mean          :  35.2490
  saturation          :   1.0187
  z_norm              :   4.6930
  u_norm              :  10.7126
  hidden_norm         :  12.6561
  lambda_gae          :   0.5000
  action_var          :   0.1815
  log_det             : -14.1224
  effective_rank      :   4.0756
  decoder_grad_norm_max:   3.3901


In [12]:
num_episodes = 20
W = 38

for _ in range(num_episodes):
    phase = 'o2' if step >= cfg.decoder_start_steps else 'tdmpc'
    obs = env.reset()
    episode = Episode(cfg, obs)
    t_ep = time.time()

    while not episode.done:
        if step < cfg.seed_steps:
            action = torch.tensor(env.action_space.sample(), dtype=torch.float32, device=agent.device)
        elif phase == 'tdmpc':
            action = agent.plan(obs, step=step, t0=episode.first)
        else:
            action, *_ = agent.CEM_in_latent(obs, step=step, sample_final_action=True)

        obs, reward, done, _ = env.step(action.cpu().numpy())
        episode += (obs, action, reward, done)
    buffer += episode
    ep_time = time.time() - t_ep

    step += cfg.episode_length
    episode_idx += 1
    env_step = int(step * cfg.action_repeat)
    print('─' * W)
    print(f'  Episode {episode_idx}   step {env_step:,}   [{phase}]')
    print('─' * W)
    print(f'  {"Reward":<16}: {episode.cumulative_reward:>8.1f}')
    print(f'  {"Horizon":<16}: {int(linear_schedule(cfg.horizon_schedule, step)):>8}')
    print(f'  {"Std":<16}: {linear_schedule(cfg.std_schedule, step):>8.3f}')
    print(f'  {"Ep time":<16}: {ep_time:>7.1f}s')


    train_metrics, update_time = {}, 0.0
    if step >= cfg.seed_steps:
        t = time.time()
        train_metrics = update_tdmpc(agent, buffer, step)
        update_time = time.time() - t

    dec_metrics = {}
    decoder_time = 0.0
    if (phase == 'o2' and step >= cfg.seed_steps):
        t = time.time()
        dec_metrics = update_decoder(agent, buffer, cfg, step)
        decoder_time = time.time() - t
        for iteration, norm in sorted(dec_metrics['grad_tracker']):
            print(f'  DCEM iter {iteration} grad norm: {norm:.6f}')



    print(f'  {"Update time":<16}: {update_time:>7.1f}s')
    if dec_metrics:
        print(f'  {"Decoder time":<16}: {decoder_time:>7.1f}s')
        for k, v in dec_metrics.items():
            if k != 'grad_tracker':
                print(f'  {k:<20}: {v:>8.4f}')
    print(f'  {"Total time":<16}: {time.time() - start_time:>7.0f}s')

    if train_metrics:
        for k, v in train_metrics.items():
            print(f'  {k:<20}: {v:>8.4f}')



──────────────────────────────────────
  Episode 21   step 21,000   [o2]
──────────────────────────────────────
  Reward          :    372.7
  Horizon         :        4
  Std             :    0.122
  Ep time         :     5.5s
  DCEM iter 0 grad norm: 0.032449
  DCEM iter 1 grad norm: 0.029307
  DCEM iter 2 grad norm: 0.026720
  DCEM iter 3 grad norm: 0.025519
  DCEM iter 4 grad norm: 0.025511
  Update time     :    28.0s
  Decoder time    :    12.9s
  decoder_loss        : -26.1457
  value_cost          : -26.2549
  diversity_cost      :   0.1093
  diversity_alpha     :   0.0109
  decoder_grad_norm   :   1.2528
  value_mean          :  35.7510
  saturation          :   2.0779
  z_norm              :   5.0621
  u_norm              :  10.6504
  hidden_norm         :  10.9323
  lambda_gae          :   0.5000
  action_var          :   0.4048
  log_det             :  -9.9911
  effective_rank      :   4.1965
  decoder_grad_norm_max:   2.8253
  Total time      :     871s
  consistency_loss 

latent_action_dim: 128
decoder_init: null
use_latent_state: true
dcem_batch_size: 128
decoder_updates: 80
told_updates: 500
decoder_start_steps: 0
exp_name: default
latent_num_samples: 32
latent_num_elites: 8
dcem_sampling_n: 5000
task: cheetah-run
modality: state
action_repeat: 4
discount: 0.99
episode_length: 250
train_steps: 10000
iterations: 5
num_samples: 512
num_elites: 64
mixture_coef: 0.05
min_std: 0.05
temperature: 0.5
momentum: 0.1
batch_size: 512
...
lambda_gae: 0.5
diversity_coeff: 0.01
use_raw_obs: true

Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Ready
Target loaded
Loaded model from models/cheetah-run20k/model.pt
Loaded buffer from models/cheetah-run20k/replay_buffer.pth
──────────────────────────────────────
  Episode 1   step 1,000   [o2]
──────────────────────────────────────
  Reward          :    174.0
  Horizon         :        5
  Std             :    0.050
  Ep time         :     5.3s
NO TDMPC TRAINING
True
True
True
True
  DCEM iter 0 grad norm: 0.034029
  DCEM iter 1 grad norm: 0.028514
  DCEM iter 2 grad norm: 0.025070
  DCEM iter 3 grad norm: 0.024451
  DCEM iter 4 grad norm: 0.025993
  Update time     :    26.6s
  Decoder time    :    10.3s
  decoder_loss        : -27.2060
  decoder_grad_norm   :   1.0448
  value_mean          :  38.0233
  saturation          :   1.2051
  z_norm              :   4.8344
  u_norm              :  10.6033
  hidden_norm         :  12.6030
  lambda_gae          :   0.5000
  action_var          :   0.1623
  log_det             : -17.0808
  effective_rank      :   3.4664
  decoder_grad_norm_max:   1.6666
  Total time      :      81s
  consistency_loss    :   0.0384
  reward_loss         :   0.0049
  value_loss          :   0.7243
  pi_loss             : -69.6703
  total_loss          :   0.1517
  weighted_loss       :   0.0946
  grad_norm           :   0.4016
──────────────────────────────────────
  Episode 2   step 2,000   [o2]
──────────────────────────────────────
  Reward          :    233.9
  Horizon         :        5
  Std             :    0.050
  Ep time         :     5.4s
NO TDMPC TRAINING
True
True
True
True
  DCEM iter 0 grad norm: 0.024660
  DCEM iter 1 grad norm: 0.021624
  DCEM iter 2 grad norm: 0.019968
  DCEM iter 3 grad norm: 0.019635
  DCEM iter 4 grad norm: 0.021043
  Update time     :    26.6s
  Decoder time    :    10.3s
  decoder_loss        : -27.7008
  decoder_grad_norm   :   1.1728
  value_mean          :  42.9238
  saturation          :   1.1760
  z_norm              :   4.8589
  u_norm              :  10.6095
  hidden_norm         :  11.8431
  lambda_gae          :   0.5000
  action_var          :   0.1759
  log_det             : -16.6748
  effective_rank      :   3.2665
  decoder_grad_norm_max:   2.3624
  Total time      :     124s
  consistency_loss    :   0.0554
  reward_loss         :   0.0069
  value_loss          :   1.5842
  pi_loss             : -76.5471
  total_loss          :   0.2726
  weighted_loss       :   0.1652
  grad_norm           :   0.7321
──────────────────────────────────────
  Episode 3   step 3,000   [o2]
──────────────────────────────────────
  Reward          :    224.6
  Horizon         :        5
  Std             :    0.050
  Ep time         :     5.4s
NO TDMPC TRAINING
True
True
True
True
  DCEM iter 0 grad norm: 0.027179
  DCEM iter 1 grad norm: 0.025092
  DCEM iter 2 grad norm: 0.024399
  DCEM iter 3 grad norm: 0.024555
  DCEM iter 4 grad norm: 0.026616
  Update time     :    26.8s
  Decoder time    :    10.4s
  decoder_loss        : -31.0670
  decoder_grad_norm   :   1.3074
  value_mean          :  45.0308
  saturation          :   1.3116
  z_norm              :   5.2360
  u_norm              :  10.6190
  hidden_norm         :  12.3819
  lambda_gae          :   0.5000
  action_var          :   0.1816
  log_det             : -16.8747
  effective_rank      :   3.1880
  decoder_grad_norm_max:   2.1113
  Total time      :     166s
  consistency_loss    :   0.0694
  reward_loss         :   0.0076
  value_loss          :   2.2425
  pi_loss             : -83.2208
  total_loss          :   0.3668
  weighted_loss       :   0.2194
  grad_norm           :   0.8341
──────────────────────────────────────
  Episode 4   step 4,000   [o2]
──────────────────────────────────────
  Reward          :    324.0
  Horizon         :        5
  Std             :    0.050
  Ep time         :     5.3s
NO TDMPC TRAINING
True
True
True
True
  DCEM iter 0 grad norm: 0.037471
  DCEM iter 1 grad norm: 0.032587
  DCEM iter 2 grad norm: 0.030253
  DCEM iter 3 grad norm: 0.029060
  DCEM iter 4 grad norm: 0.031731
  Update time     :    26.5s
  Decoder time    :    10.6s
  decoder_loss        : -33.8067
  decoder_grad_norm   :   1.4118
  value_mean          :  50.7048
  saturation          :   1.3916
  z_norm              :   5.3916
  u_norm              :  10.4674
  hidden_norm         :  12.3837
  lambda_gae          :   0.5000
  action_var          :   0.1728
  log_det             : -17.7975
  effective_rank      :   3.1156
  decoder_grad_norm_max:   2.1672
  Total time      :     209s
  consistency_loss    :   0.0670
  reward_loss         :   0.0063
  value_loss          :   2.0226
  pi_loss             : -90.9098
  total_loss          :   0.3394
  weighted_loss       :   0.2022
  grad_norm           :   0.8669
──────────────────────────────────────
  Episode 5   step 5,000   [o2]
──────────────────────────────────────
  Reward          :     70.4
  Horizon         :        5
  Std             :    0.050
  Ep time         :     5.3s
NO TDMPC TRAINING
True
True
True
True
  DCEM iter 0 grad norm: 0.032012
  DCEM iter 1 grad norm: 0.028933
  DCEM iter 2 grad norm: 0.027565
  DCEM iter 3 grad norm: 0.028027
  DCEM iter 4 grad norm: 0.030397
  Update time     :    26.7s
  Decoder time    :    10.6s
  decoder_loss        : -35.8774
  decoder_grad_norm   :   1.4770
  value_mean          :  54.0025
  saturation          :   1.3205
  z_norm              :   5.5880
  u_norm              :  10.4605
  hidden_norm         :  12.0368
  lambda_gae          :   0.5000
  action_var          :   0.1790
  log_det             : -17.4742
  effective_rank      :   3.1448
  decoder_grad_norm_max:   2.3276
  Total time      :     251s
  consistency_loss    :   0.0678
  reward_loss         :   0.0055
  value_loss          :   1.8247
  pi_loss             : -97.2106
  total_loss          :   0.3208
  weighted_loss       :   0.1930
  grad_norm           :   0.8670
──────────────────────────────────────
  Episode 6   step 6,000   [o2]
──────────────────────────────────────
  Reward          :    289.8
  Horizon         :        5
  Std             :    0.050
  Ep time         :     5.4s
NO TDMPC TRAINING
True
True
True
True
  DCEM iter 0 grad norm: 0.033129
  DCEM iter 1 grad norm: 0.028101
  DCEM iter 2 grad norm: 0.026184
  DCEM iter 3 grad norm: 0.025834
  DCEM iter 4 grad norm: 0.028168
  Update time     :    26.9s
  Decoder time    :    10.6s
  decoder_loss        : -37.9272
  decoder_grad_norm   :   1.4448
  value_mean          :  58.4014
  saturation          :   1.2968
  z_norm              :   5.8657
  u_norm              :  10.5058
  hidden_norm         :  11.9440
  lambda_gae          :   0.5000
  action_var          :   0.1717
  log_det             : -17.9093
  effective_rank      :   3.1414
  decoder_grad_norm_max:   2.2844
  Total time      :     294s
  consistency_loss    :   0.0776
  reward_loss         :   0.0064
  value_loss          :   2.3676
  pi_loss             : -106.1258
  total_loss          :   0.3952
  weighted_loss       :   0.2307
  grad_norm           :   0.9880
──────────────────────────────────────
  Episode 7   step 7,000   [o2]
──────────────────────────────────────
  Reward          :    335.4
  Horizon         :        5
  Std             :    0.050
  Ep time         :     5.5s
NO TDMPC TRAINING
True
True
True
True
  DCEM iter 0 grad norm: 0.033778
  DCEM iter 1 grad norm: 0.030888
  DCEM iter 2 grad norm: 0.029653
  DCEM iter 3 grad norm: 0.030150
  DCEM iter 4 grad norm: 0.033256
  Update time     :    26.9s
  Decoder time    :    10.5s
  decoder_loss        : -44.4886
  decoder_grad_norm   :   1.5197
  value_mean          :  60.6876
  saturation          :   1.5267
  z_norm              :   6.3586
  u_norm              :  10.5436
  hidden_norm         :  12.6550
  lambda_gae          :   0.5000
  action_var          :   0.1762
  log_det             : -18.6999
  effective_rank      :   3.0712
  decoder_grad_norm_max:   2.2023
  Total time      :     337s
  consistency_loss    :   0.0647
  reward_loss         :   0.0041
  value_loss          :   1.1495
  pi_loss             : -113.9707
  total_loss          :   0.2463
  weighted_loss       :   0.1524
  grad_norm           :   0.6833
──────────────────────────────────────
  Episode 8   step 8,000   [o2]
──────────────────────────────────────
  Reward          :    427.7
  Horizon         :        5
  Std             :    0.050
  Ep time         :     5.4s
NO TDMPC TRAINING
True
True
True
True
  DCEM iter 0 grad norm: 0.038752
  DCEM iter 1 grad norm: 0.035558
  DCEM iter 2 grad norm: 0.033817
  DCEM iter 3 grad norm: 0.033712
  DCEM iter 4 grad norm: 0.038174
  Update time     :    26.8s
  Decoder time    :    10.5s
  decoder_loss        : -47.7557
  decoder_grad_norm   :   1.5278
  value_mean          :  65.4441
  saturation          :   1.5579
  z_norm              :   6.5376
  u_norm              :  10.4920
  hidden_norm         :  12.8318
  lambda_gae          :   0.5000
  action_var          :   0.1773
  log_det             : -18.6138
  effective_rank      :   3.1005
  decoder_grad_norm_max:   2.3507
  Total time      :     380s
  consistency_loss    :   0.0536
  reward_loss         :   0.0027
  value_loss          :   0.5831
  pi_loss             : -121.9680
  total_loss          :   0.1668
  weighted_loss       :   0.1106
  grad_norm           :   0.6189
──────────────────────────────────────
  Episode 9   step 9,000   [o2]
──────────────────────────────────────
  Reward          :    425.6
  Horizon         :        5
  Std             :    0.050
  Ep time         :     5.3s
NO TDMPC TRAINING
True
True
True
True
  DCEM iter 0 grad norm: 0.030057
  DCEM iter 1 grad norm: 0.027707
  DCEM iter 2 grad norm: 0.026860
  DCEM iter 3 grad norm: 0.026943
  DCEM iter 4 grad norm: 0.028619
  Update time     :    26.7s
  Decoder time    :    10.5s
  decoder_loss        : -50.4553
  decoder_grad_norm   :   1.3747
  value_mean          :  69.6318
  saturation          :   1.5651
  z_norm              :   6.8402
  u_norm              :  10.4370
  hidden_norm         :  12.7688
  lambda_gae          :   0.5000
  action_var          :   0.1759
  log_det             : -18.6458
  effective_rank      :   3.0827
  decoder_grad_norm_max:   2.5699
  Total time      :     422s
  consistency_loss    :   0.0650
  reward_loss         :   0.0044
  value_loss          :   1.2171
  pi_loss             : -132.3777
  total_loss          :   0.2540
  weighted_loss       :   0.1590
  grad_norm           :   0.9404
──────────────────────────────────────
  Episode 10   step 10,000   [o2]
──────────────────────────────────────
  Reward          :    387.9
  Horizon         :        5
  Std             :    0.050
  Ep time         :     5.4s
NO TDMPC TRAINING
True
True
True
True
  DCEM iter 0 grad norm: 0.041912
  DCEM iter 1 grad norm: 0.032123
  DCEM iter 2 grad norm: 0.027530
  DCEM iter 3 grad norm: 0.028195
  DCEM iter 4 grad norm: 0.031001
  Update time     :    26.6s
  Decoder time    :    10.6s
  decoder_loss        : -52.9965
  decoder_grad_norm   :   1.3701
  value_mean          :  74.3978
  saturation          :   1.5168
  z_norm              :   6.9907
  u_norm              :  10.3695
  hidden_norm         :  12.4680
  lambda_gae          :   0.5000
  action_var          :   0.1709
  log_det             : -18.6590
  effective_rank      :   3.1128
  decoder_grad_norm_max:   2.0570
  Total time      :     465s
  consistency_loss    :   0.0672
  reward_loss         :   0.0045
  value_loss          :   1.1990
  pi_loss             : -141.6663
  total_loss          :   0.2566
  weighted_loss       :   0.1603
  grad_norm           :   0.9545
──────────────────────────────────────
  Episode 11   step 11,000   [o2]
──────────────────────────────────────
  Reward          :    415.9
  Horizon         :        5
  Std             :    0.050
  Ep time         :     5.4s
NO TDMPC TRAINING
True
True
True
True
  DCEM iter 0 grad norm: 0.032727
  DCEM iter 1 grad norm: 0.031728
  DCEM iter 2 grad norm: 0.031045
  DCEM iter 3 grad norm: 0.031795
  DCEM iter 4 grad norm: 0.035046
  Update time     :    26.5s
  Decoder time    :    10.4s
  decoder_loss        : -57.2412
  decoder_grad_norm   :   1.3878
  value_mean          :  80.5999
  saturation          :   1.4520
  z_norm              :   7.0029
  u_norm              :  10.3599
  hidden_norm         :  12.2682
  lambda_gae          :   0.5000
  action_var          :   0.1684
  log_det             : -18.4553
  effective_rank      :   3.1429
  decoder_grad_norm_max:   2.4219
  Total time      :     507s
  consistency_loss    :   0.0626
  reward_loss         :   0.0038
  value_loss          :   0.9835
  pi_loss             : -152.3252
  total_loss          :   0.2254
  weighted_loss       :   0.1436
  grad_norm           :   0.9178
──────────────────────────────────────
  Episode 12   step 12,000   [o2]
──────────────────────────────────────
  Reward          :    455.5
  Horizon         :        5
  Std             :    0.050
  Ep time         :     5.3s
NO TDMPC TRAINING
True
True
True
True
  DCEM iter 0 grad norm: 0.055185
  DCEM iter 1 grad norm: 0.043585
  DCEM iter 2 grad norm: 0.037082
  DCEM iter 3 grad norm: 0.033931
  DCEM iter 4 grad norm: 0.034926
  Update time     :    26.5s
  Decoder time    :    10.5s
  decoder_loss        : -61.4711
  decoder_grad_norm   :   1.4409
  value_mean          :  83.9537
  saturation          :   1.4172
  z_norm              :   7.3702
  u_norm              :  10.3241
  hidden_norm         :  12.0540
  lambda_gae          :   0.5000
  action_var          :   0.1678
  log_det             : -18.1800
  effective_rank      :   3.1714
  decoder_grad_norm_max:   2.3338
  Total time      :     550s
  consistency_loss    :   0.0598
  reward_loss         :   0.0034
  value_loss          :   0.8492
  pi_loss             : -159.7109
  total_loss          :   0.2062
  weighted_loss       :   0.1337
  grad_norm           :   0.8960
──────────────────────────────────────
  Episode 13   step 13,000   [o2]
──────────────────────────────────────
  Reward          :     80.1
  Horizon         :        5
  Std             :    0.050
  Ep time         :     5.4s
NO TDMPC TRAINING
True
True
True
True
  DCEM iter 0 grad norm: 0.035450
  DCEM iter 1 grad norm: 0.031001
  DCEM iter 2 grad norm: 0.027772
  DCEM iter 3 grad norm: 0.029644
  DCEM iter 4 grad norm: 0.035110
  Update time     :    26.9s
  Decoder time    :    10.4s
  decoder_loss        : -62.9611
  decoder_grad_norm   :   1.3350
  value_mean          :  86.9492
  saturation          :   1.4937
  z_norm              :   7.5313
  u_norm              :  10.3550
  hidden_norm         :  12.2928
  lambda_gae          :   0.5000
  action_var          :   0.1617
  log_det             : -19.0245
  effective_rank      :   3.0858
  decoder_grad_norm_max:   2.6044
  Total time      :     592s
  consistency_loss    :   0.0864
  reward_loss         :   0.0073
  value_loss          :   1.6240
  pi_loss             : -163.7968
  total_loss          :   0.3388
  weighted_loss       :   0.2051
  grad_norm           :   1.0937
──────────────────────────────────────
  Episode 14   step 14,000   [o2]
──────────────────────────────────────
  Reward          :    468.4
  Horizon         :        5
  Std             :    0.050
  Ep time         :     5.4s
NO TDMPC TRAINING
True
True
True
True
  DCEM iter 0 grad norm: 0.020650
  DCEM iter 1 grad norm: 0.019406
  DCEM iter 2 grad norm: 0.018963
  DCEM iter 3 grad norm: 0.020368
  DCEM iter 4 grad norm: 0.022998
  Update time     :    26.8s
  Decoder time    :    10.4s
  decoder_loss        : -64.2015
  decoder_grad_norm   :   1.2937
  value_mean          :  90.0597
  saturation          :   1.5939
  z_norm              :   7.9395
  u_norm              :  10.2894
  hidden_norm         :  12.7006
  lambda_gae          :   0.5000
  action_var          :   0.1607
  log_det             : -19.8118
  effective_rank      :   3.0074
  decoder_grad_norm_max:   3.2860
  Total time      :     635s
  consistency_loss    :   0.0600
  reward_loss         :   0.0030
  value_loss          :   0.7191
  pi_loss             : -172.5190
  total_loss          :   0.1934
  weighted_loss       :   0.1255
  grad_norm           :   0.9220
──────────────────────────────────────
  Episode 15   step 15,000   [o2]
──────────────────────────────────────
  Reward          :    204.1
  Horizon         :        5
  Std             :    0.050
  Ep time         :     5.3s
NO TDMPC TRAINING
True
True
True
True
  DCEM iter 0 grad norm: 0.036586
  DCEM iter 1 grad norm: 0.032282
  DCEM iter 2 grad norm: 0.029771
  DCEM iter 3 grad norm: 0.028031
  DCEM iter 4 grad norm: 0.030174
  Update time     :    26.8s
  Decoder time    :    10.5s
  decoder_loss        : -65.9715
  decoder_grad_norm   :   1.2267
  value_mean          :  91.9206
  saturation          :   1.6603
  z_norm              :   8.2121
  u_norm              :  10.2550
  hidden_norm         :  12.8949
  lambda_gae          :   0.5000
  action_var          :   0.1604
  log_det             : -20.2147
  effective_rank      :   2.9703
  decoder_grad_norm_max:   2.5205
  Total time      :     677s
  consistency_loss    :   0.1080
  reward_loss         :   0.0059
  value_loss          :   2.2471
  pi_loss             : -174.3519
  total_loss          :   0.4437
  weighted_loss       :   0.2533
  grad_norm           :   1.1349
──────────────────────────────────────
  Episode 16   step 16,000   [o2]
──────────────────────────────────────
  Reward          :    487.7
  Horizon         :        5
  Std             :    0.050
  Ep time         :     5.4s
NO TDMPC TRAINING
True
True
True
True
  DCEM iter 0 grad norm: 0.019768
  DCEM iter 1 grad norm: 0.019015
  DCEM iter 2 grad norm: 0.018488
  DCEM iter 3 grad norm: 0.019520
  DCEM iter 4 grad norm: 0.021749
  Update time     :    26.4s
  Decoder time    :    10.5s
  decoder_loss        : -65.4471
  decoder_grad_norm   :   1.2181
  value_mean          :  95.7628
  saturation          :   1.6704
  z_norm              :   8.6316
  u_norm              :  10.2603
  hidden_norm         :  12.8037
  lambda_gae          :   0.5000
  action_var          :   0.1666
  log_det             : -19.9760
  effective_rank      :   2.9500
  decoder_grad_norm_max:   3.5410
  Total time      :     720s
  consistency_loss    :   0.0675
  reward_loss         :   0.0031
  value_loss          :   0.8123
  pi_loss             : -182.1388
  total_loss          :   0.2179
  weighted_loss       :   0.1401
  grad_norm           :   0.9591
──────────────────────────────────────
  Episode 17   step 17,000   [o2]
──────────────────────────────────────
  Reward          :    546.7
  Horizon         :        5
  Std             :    0.050
  Ep time         :     5.4s
NO TDMPC TRAINING
True
True
True
True
  DCEM iter 0 grad norm: 0.028251
  DCEM iter 1 grad norm: 0.024801
  DCEM iter 2 grad norm: 0.022904
  DCEM iter 3 grad norm: 0.023177
  DCEM iter 4 grad norm: 0.024912
  Update time     :    26.7s
  Decoder time    :    10.4s
  decoder_loss        : -70.6205
  decoder_grad_norm   :   1.2733
  value_mean          :  99.3863
  saturation          :   1.7660
  z_norm              :   9.2637
  u_norm              :  10.2309
  hidden_norm         :  13.0116
  lambda_gae          :   0.5000
  action_var          :   0.1651
  log_det             : -20.5075
  effective_rank      :   2.8988
  decoder_grad_norm_max:   4.6404
  Total time      :     762s
  consistency_loss    :   0.0758
  reward_loss         :   0.0031
  value_loss          :   0.9907
  pi_loss             : -187.6442
  total_loss          :   0.2522
  weighted_loss       :   0.1580
  grad_norm           :   1.0423
──────────────────────────────────────
  Episode 18   step 18,000   [o2]
──────────────────────────────────────
  Reward          :    488.0
  Horizon         :        5
  Std             :    0.050
  Ep time         :     5.4s
NO TDMPC TRAINING
True
True
True
True
  DCEM iter 0 grad norm: 0.024968
  DCEM iter 1 grad norm: 0.023475
  DCEM iter 2 grad norm: 0.023692
  DCEM iter 3 grad norm: 0.024799
  DCEM iter 4 grad norm: 0.027757
  Update time     :    26.7s
  Decoder time    :    10.5s
  decoder_loss        : -74.6697
  decoder_grad_norm   :   1.3625
  value_mean          : 102.8933
  saturation          :   1.7499
  z_norm              :   9.4865
  u_norm              :  10.2445
  hidden_norm         :  13.0280
  lambda_gae          :   0.5000
  action_var          :   0.1668
  log_det             : -20.3101
  effective_rank      :   2.8989
  decoder_grad_norm_max:   3.7859
  Total time      :     805s
  consistency_loss    :   0.0665
  reward_loss         :   0.0025
  value_loss          :   0.7274
  pi_loss             : -198.4788
  total_loss          :   0.2069
  weighted_loss       :   0.1318
  grad_norm           :   1.1789
──────────────────────────────────────
  Episode 19   step 19,000   [o2]
──────────────────────────────────────
  Reward          :     30.4
  Horizon         :        5
  Std             :    0.050
  Ep time         :     5.3s
NO TDMPC TRAINING
True
True
True
True
  DCEM iter 0 grad norm: 0.024439
  DCEM iter 1 grad norm: 0.022985
  DCEM iter 2 grad norm: 0.023234
  DCEM iter 3 grad norm: 0.024557
  DCEM iter 4 grad norm: 0.027652
  Update time     :    26.7s
  Decoder time    :    10.4s
  decoder_loss        : -75.4523
  decoder_grad_norm   :   1.2981
  value_mean          : 104.6303
  saturation          :   1.7314
  z_norm              :   9.7617
  u_norm              :  10.1311
  hidden_norm         :  12.9027
  lambda_gae          :   0.5000
  action_var          :   0.1727
  log_det             : -20.1568
  effective_rank      :   2.9249
  decoder_grad_norm_max:   2.2648
  Total time      :     847s
  consistency_loss    :   0.1437
  reward_loss         :   0.0064
  value_loss          :   2.6266
  pi_loss             : -198.7172
  total_loss          :   0.5533
  weighted_loss       :   0.3131
  grad_norm           :   1.4930
──────────────────────────────────────
  Episode 20   step 20,000   [o2]
──────────────────────────────────────
  Reward          :     97.1
  Horizon         :        5
  Std             :    0.050
  Ep time         :     5.4s

In [14]:


eval_metrics = evaluate_agent(
    env,
    agent,
    cfg,
    step=step,
    n_episodes=1,
    use_latent=True,
    save_dir='eval_videos',
    video_mode='first'   # or 'first'
)


Episode 1/1: Reward = 237.935
🎥 Saved first episode video: eval_videos/videos/eval_step10000_ep001.mp4

Evaluation Summary — Step 10000
-------------------------
Mean Reward: 237.935
Std Reward:  0.000


In [16]:
import torch
del agent, buffer  # or whatever large objects
torch.cuda.empty_cache()

In [ ]:
import torch.nn.utils as utils
import pandas as pd
#PAIRWISE DISTANCES LOSS, DIAGONAL ELEMENTS INCLUDED
def update_decoder_DDPG(self, obs, u_mean, horizon, weights=None, log_det_loss=None, u_std=None):
    self.action_dec_optim.zero_grad()

    k     = getattr(self.cfg, 'diversity_k_samples', 3)
    #sigma = getattr(self.cfg, 'diversity_sigma', 0.1)
    B     = u_mean.shape[0]

    z      = self.model.h(obs).detach()
    z_norm = z.norm(dim=-1).mean().item()
    u_norm = u_mean.norm(dim=-1).mean().item()

    min_std   = self.std
    noise     = torch.randn(k, B, u_mean.shape[-1], device=z.device)
    u_samples = u_mean.unsqueeze(0) + u_std.clamp(min=min_std).unsqueeze(0) * noise


    u_flat    = u_samples.permute(1, 0, 2).reshape(B * k, -1)
    z_rep     = z.repeat_interleave(k, dim=0)

    seqs, pretanh = self.model.decode_sequence(u_flat, z_rep, return_pretanh=True)
    saturation    = pretanh.abs().mean().item()

    #values = self.estimate_value_with_grad(z_rep, seqs, horizon).nan_to_num(0).squeeze(-1)

    values = self.estimate_value_GAE(z_rep, seqs, horizon).nan_to_num(0).squeeze(-1)

    seqs_bk = seqs[:horizon].reshape(horizon, B, k, -1).permute(1, 2, 0, 3).flatten(2)

    values_bk = values.reshape(B, k)

    v         = (values_bk - values_bk.min(dim=1, keepdim=True).values.detach()).clamp(min=1e-6)
    #v         = values_bk.clamp(min=1e-6)
    #v = values_bk.clamp(min=1e-6)
    dists     = torch.cdist(seqs_bk, seqs_bk)
    sigma     = dists.detach().mean().clamp(min=1e-6)


    scaled_dists = 1 + dists/sigma    # linear growth, starts at 1 like exp
    #scaled_dists = torch.exp((dists/sigma).clamp(max=3))

    #v_weights    = v.unsqueeze(2) * v.unsqueeze(1)
    v_weights = v.sqrt().unsqueeze(2) * v.sqrt().unsqueeze(1)
    #v_weights = v.sqrt().unsqueeze(2) * v.sqrt().unsqueeze(1)
    mask      = torch.triu(torch.ones(k, k, device=z.device, dtype=torch.bool), diagonal=0)
    cost      = -(v_weights * scaled_dists)[: , mask].mean()
    cost.backward()


    k = v.shape[1]

    cost_matrix         = (v_weights * scaled_dists).detach().mean(dim=0).cpu().numpy()
    dist_matrix         = dists.detach().mean(dim=0).cpu().numpy()
    dist_score_matrix = scaled_dists.detach().mean(dim=0).cpu().numpy()
    val_matrix          = v_weights.detach().mean(dim=0).cpu().numpy()

    df_cost       = pd.DataFrame(cost_matrix.round(3), index=[f"s{i}" for i in range(k)], columns=[f"s{j}" for j in range(k)])
    df_dist       = pd.DataFrame(dist_matrix.round(3), index=[f"s{i}" for i in range(k)], columns=[f"s{j}" for j in range(k)])
    df_dist_score = pd.DataFrame(dist_score_matrix.round(3), index=[f"s{i}" for i in range(k)], columns=[f"s{j}" for j in range(k)])
    df_val        = pd.DataFrame(val_matrix.round(3),  index=[f"s{i}" for i in range(k)], columns=[f"s{j}" for j in range(k)])

    print("=== distances ===");       print(df_dist.to_string())
    print("=== exp_dists ===");       print(df_dist_score.to_string())
    print("=== value weights ===");   print(df_val.to_string())
    print("=== cost matrix ===");     print(df_cost.to_string())

    seqs_bk_unflat = seqs[:horizon].reshape(horizon, B, k, -1).permute(1, 2, 0, 3)

    per_step_dists = []
    for t in range(horizon):
        d_t = torch.cdist(seqs_bk_unflat[:, :, t, :], seqs_bk_unflat[:, :, t, :])  # [B, k, k]
        mask = torch.triu(torch.ones(k, k, dtype=torch.bool, device=d_t.device), diagonal=1)
        per_step_dists.append(d_t[:, mask].mean().item())

    for t, d in enumerate(per_step_dists):
        print(f"  t={t}: mean_dist = {d:.4f}")
    print(f"  total (flattened): {dists.detach()[:, mask].mean().item():.4f}")

    grad_norm = torch.sqrt(sum(
        p.grad.norm() ** 2
        for p in self.model._action_decoder.parameters() if p.grad is not None
    ))
    dec_grad_clip = getattr(self.cfg, 'dec_grad_clip_norm', None)
    if dec_grad_clip:
        utils.clip_grad_norm_(self.model._action_decoder.parameters(), dec_grad_clip)
    self.action_dec_optim.step()

    return {
        'decoder_loss':      cost.item(),
        'decoder_grad_norm': grad_norm.item(),
        'value_mean':        values_bk.mean().item(),
        'saturation':        saturation,
        'z_norm':            z_norm,
        'u_norm':            u_norm,
        'hidden_norm':       self.model._action_decoder._hidden_norm,
    }
setattr(TDMPC_O2, "update_decoder_DDPG", update_decoder_DDPG)

In [ ]:
import torch.nn.utils as utils
import pandas as pd
#DPP LOSS
def update_decoder_DDPG(self, obs, u_mean, horizon, weights=None, log_det_loss=None):
    self.action_dec_optim.zero_grad()

    k     = getattr(self.cfg, 'diversity_k_samples', 8)
    sigma_noise = getattr(self.cfg, 'diversity_sigma', 0.05)
    B     = u_mean.shape[0]

    z      = self.model.h(obs).detach()
    z_norm = z.norm(dim=-1).mean().item()
    u_norm = u_mean.norm(dim=-1).mean().item()

    noise     = torch.randn(k, B, u_mean.shape[-1], device=z.device)
    u_samples = u_mean.unsqueeze(0) + sigma_noise * noise
    u_flat    = u_samples.permute(1, 0, 2).reshape(B * k, -1)
    z_rep     = z.repeat_interleave(k, dim=0)

    seqs, pretanh = self.model.decode_sequence(u_flat, z_rep, return_pretanh=True)
    saturation    = pretanh.abs().mean().item()

    values = self.estimate_value_with_grad(z_rep, seqs, horizon).nan_to_num(0).squeeze(-1)

    seqs_bk   = seqs.reshape(horizon, B, k, -1).permute(1, 2, 0, 3).flatten(2)
    values_bk = values.reshape(B, k)

    v     = (values_bk - values_bk.min(dim=1, keepdim=True).values.detach()).clamp(min=1e-6)
    dists = torch.cdist(seqs_bk, seqs_bk)
    sigma = getattr(self.cfg, 'dpp_sigma', 1.0)
    K     = torch.exp(-dists**2 / (2 * sigma**2))
    L     = v.unsqueeze(2) * K * v.unsqueeze(1)
    L     = L + 1e-4 * torch.eye(k, device=z.device).unsqueeze(0)
    sign, logdet = torch.linalg.slogdet(L)
    cost  = -(sign * logdet).mean()

    cost.backward()

    grad_norm = torch.sqrt(sum(
        p.grad.norm() ** 2
        for p in self.model._action_decoder.parameters() if p.grad is not None
    ))
    dec_grad_clip = getattr(self.cfg, 'dec_grad_clip_norm', None)
    if dec_grad_clip:
        utils.clip_grad_norm_(self.model._action_decoder.parameters(), dec_grad_clip)
    self.action_dec_optim.step()

    with torch.no_grad():
        L_print = L.detach().mean(dim=0)
        df = pd.DataFrame(L_print.cpu().numpy().round(3),
                          index=[f"s{i}" for i in range(k)],
                          columns=[f"s{j}" for j in range(k)])
        print(df.to_string())

    return {
        'decoder_loss':      cost.item(),
        'decoder_grad_norm': grad_norm.item(),
        'value_mean':        values_bk.mean().item(),
        'saturation':        saturation,
        'z_norm':            z_norm,
        'u_norm':            u_norm,
        'hidden_norm':       self.model._action_decoder._hidden_norm,
    }
setattr(TDMPC_O2, "update_decoder_DDPG", update_decoder_DDPG)

In [ ]:
import torch.nn.utils as utils
def action_decoder_DDPG_update_v2(self, obs, u_mean, u_std, horizon, weights=None):
    """
    DDPG-style decoder update with entropy regularization and saturation penalty.

    Two saturation penalty options are available — select by uncommenting:
      Option 1 (inline jacobian penalty): fast, uses pretanh values directly.
      Option 2 (saturation_loss):         sampled, captures full distribution.

    Args:
        obs:     [B, obs_dim] observation batch from the replay buffer.
        u_mean:  [B, latent_action_dim] differentiable latent action mean
                 obtained from DCEMethod(update_mode=True).
        u_std:   [B, latent_action_dim] differentiable latent action std
                 obtained from DCEMethod(update_mode=True).
        horizon: int planning horizon.

    Returns:
        dict with keys: decoder_loss, decoder_grad_norm, saturation
    """
    # Are any TOLD params mistakenly receiving grads?
    for name, p in agent.model.named_parameters():
        if '_action_decoder' not in name and '_V' not in name:
            assert not p.requires_grad, f"TOLD param not frozen: {name}"

    self.action_dec_optim.zero_grad()

    z                 = self.model.h(obs).detach()
    sequence, pretanh = self.model.decode_sequence_pretanh(u_mean, z)
    value             = self.estimate_value_with_grad(z, sequence, horizon).nan_to_num(0).squeeze(-1)
    saturation        = pretanh.abs().mean().item()

    # --- Option 1: inline jacobian penalty [B], subtracted per batch element ---
    jacobian_penalty  = -torch.log(1 - sequence.pow(2) + 1e-6).sum(-1).mean(0)  # [B]
    per_sample_cost   = -(value - self.cfg.saturation_coeff * jacobian_penalty)
    if weights is not None:
        cost = (per_sample_cost * weights).mean()
    else:
        cost = per_sample_cost.mean()

    # --- Option 2: saturation_loss (sampled across distribution) [B] ---
    #saturation_coeff  = getattr(self.cfg, 'saturation_coeff', 0.0)
    #sat_loss          = self.saturation_loss(u_mean, u_std, z) if saturation_coeff > 0 else 0.0
    #per_sample_cost   = -(value - saturation_coeff * sat_loss)
    #if weights is not None:
    #    cost = (per_sample_cost * weights).mean()
    #else:
    #    cost = per_sample_cost.mean()
    cost.backward()

    grad_norm = torch.sqrt(sum(
        p.grad.norm() ** 2
        for p in self.model._action_decoder.parameters() if p.grad is not None
    ))
    dec_grad_clip = getattr(self.cfg, 'dec_grad_clip_norm', None)
    if dec_grad_clip:
        utils.clip_grad_norm_(self.model._action_decoder.parameters(), max_norm=dec_grad_clip)
    self.action_dec_optim.step()

    return {'decoder_loss': cost.item(), 'decoder_grad_norm': grad_norm.item(), 'saturation': saturation}



setattr(TDMPC_O2, "action_decoder_DDPG_update_v2", action_decoder_DDPG_update_v2)


In [ ]:
from o2.eval_utils import evaluate_agent

metrics = evaluate_agent(
    env, agent, cfg,
    step=0,
    n_episodes=1,
    save_dir='./eval_output',
    video_mode='first',
)
for k, v in metrics.items():
    print(f'  {k:<20}: {v:>8.4f}')

Episode 1/1: Reward = 67.106
🎥 Saved first episode video: ./eval_output/videos/eval_step0_ep001.mp4

Evaluation Summary — Step 0
-------------------------
Mean Reward: 67.106
Std Reward:  0.000
  step                :   0.0000
  mean_reward         :  67.1056
  std_reward          :   0.0000
  mean_compute_duration:   0.0052
  episode_duration    :  22.3075


TypeError: unsupported format string passed to list.__format__